# 03 - KNN Model & Hyperparameter Tuning

Train a baseline KNN model, evaluate, and tune K.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
import pickle
import os
import sys

sys.path.append(os.path.abspath('..'))
from src.evaluate import calculate_metrics, print_classification_report, plot_confusion_matrix


### PART A — Train baseline KNN

#### 1. Load Data


In [ ]:
with open('../data/processed/processed_data.pkl', 'rb') as f:
    X_train_scaled, X_test_scaled, y_train, y_test, scaler = pickle.load(f)


#### 2 & 3. Train and Predict


In [ ]:
knn_baseline = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_baseline.fit(X_train_scaled, y_train)
y_pred = knn_baseline.predict(X_test_scaled)


#### 4 & 5. Evaluation Metrics & Confusion Matrix


In [ ]:
metrics = calculate_metrics(y_test, y_pred)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

print("\nClassification Report:")
print_classification_report(y_test, y_pred)

plot_confusion_matrix(y_test, y_pred, title="Baseline KNN (K=5) Confusion Matrix")


#### 6. Why Recall is more important than Accuracy?

In medical diagnosis (like predicting heart disease):
- **False Positive**: Telling a healthy patient they are sick (leads to more tests, anxiety, but not directly fatal).
- **False Negative**: Telling a sick patient they are healthy (they go home, miss treatment, and could have a heart attack).

Since a False Negative is much more dangerous, we want to minimize it. This means maximizing **Recall (Sensitivity)** — we want to catch as many real heart disease cases as possible, even if it lowers our precision slightly.


### PART B — Hyperparameter Tuning

#### 7 & 8. Loop K from 1 to 30


In [ ]:
k_values = range(1, 31)
accuracies = []
precisions = []
recalls = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train_scaled, y_train)
    pred = knn.predict(X_test_scaled)
    
    m = calculate_metrics(y_test, pred)
    accuracies.append(m['Accuracy'])
    precisions.append(m['Precision'])
    recalls.append(m['Recall'])


#### 9. Plot 3 separate line graphs


In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(k_values, accuracies, marker='o')
plt.title('K vs Accuracy')
plt.xlabel('K')
plt.ylabel('Accuracy')

plt.subplot(1, 3, 2)
plt.plot(k_values, precisions, marker='o', color='orange')
plt.title('K vs Precision')
plt.xlabel('K')
plt.ylabel('Precision')

plt.subplot(1, 3, 3)
plt.plot(k_values, recalls, marker='o', color='green')
plt.title('K vs Recall')
plt.xlabel('K')
plt.ylabel('Recall')

plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/knn_accuracy_curve.png')
plt.show()


#### 10. Identify Best K based on Recall


In [ ]:
best_k = k_values[np.argmax(recalls)]
print(f"Best K based on highest recall is: {best_k}")
print(f"Max Recall: {max(recalls):.4f}")


#### 11 & 12. Re-train final KNN and Save


In [ ]:
final_knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
final_knn.fit(X_train_scaled, y_train)
final_pred = final_knn.predict(X_test_scaled)

print("\n--- Final KNN Metrics ---")
final_metrics = calculate_metrics(y_test, final_pred)
for k, v in final_metrics.items():
    print(f"{k}: {v:.4f}")

plot_confusion_matrix(y_test, final_pred, title=f"Final KNN (K={best_k}) Confusion Matrix")

os.makedirs('../models', exist_ok=True)
with open('../models/knn_model.pkl', 'wb') as f:
    pickle.dump(final_knn, f)
print("Model saved to models/knn_model.pkl")
